# Análise Exploratória de Dados - Datathon Passos Mágicos

Este notebook realiza a análise exploratória dos indicadores educacionais da ONG Passos Mágicos.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set(style="whitegrid")

## 1. Carregamento dos Dados

Carregando o dataset principal para análise.

In [ ]:
# Carregando o dataset CSV (PEDE_PASSOS_DATASET_FIAP.csv)
df = pd.read_csv('../data/raw/PEDE_PASSOS_DATASET_FIAP.csv', sep=';')
df.head()

## 2. Padronização e Limpeza

Selecionando os indicadores principais e convertendo para numérico.

In [ ]:
# Indicadores principais de 2020
indicadores = ['INDE_2020', 'IDA_2020', 'IEG_2020', 'IAA_2020', 'IPS_2020', 'IPP_2020', 'IAN_2020', 'IPV_2020']

# Criando um novo dataframe apenas com os indicadores
df_eda = df[indicadores].copy()

# Renomeando para nomes padronizados
df_eda.columns = [col.split('_')[0] for col in df_eda.columns]

# Função para converter para numérico (tratando vírgulas e erros)
def to_numeric(col):
    return pd.to_numeric(col.astype(str).str.replace(',', '.'), errors='coerce')

for col in df_eda.columns:
    df_eda[col] = to_numeric(df_eda[col])

# Tratando valores ausentes (removendo linhas onde o INDE é nulo para análise de correlação)
df_eda = df_eda.dropna(subset=['INDE'])

df_eda.info()
df_eda.describe()

## 3. Distribuição dos Indicadores

Visualizando a distribuição de cada indicador através de histogramas.

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(15, 20))
axes = axes.flatten()

for i, col in enumerate(df_eda.columns):
    sns.histplot(df_eda[col], kde=True, ax=axes[i], color='skyblue')
    axes[i].set_title(f'Distribuição de {col}', fontsize=14)
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Frequência')

plt.tight_layout()
plt.show()

> **Insight:** A maioria dos indicadores apresenta uma distribuição próxima da normal, com alguns deslocamentos. O INDE, sendo um índice sintético, mostra uma concentração clara em faixas específicas de desempenho.

## 4. Matriz de Correlação

Analisando a relação entre os diferentes indicadores.

In [ ]:
plt.figure(figsize=(10, 8))
corr = df_eda.corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Matriz de Correlação dos Indicadores', fontsize=16)
plt.show()

> **Insight:** Existe uma forte correlação entre o INDE e indicadores como IDA (Indicador de Aprendizagem) e IEG (Indicador de Engajamento), sugerindo que estes são pilares fundamentais para o desenvolvimento do aluno.

## 5. Análise de Relações Específicas

Explorando a relação entre Engajamento, Aprendizagem e Autoavaliação.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# IEG vs IDA
sns.scatterplot(data=df_eda, x='IEG', y='IDA', ax=ax1, alpha=0.6)
ax1.set_title('Engajamento (IEG) vs Aprendizagem (IDA)', fontsize=14)

# IAA vs IDA
sns.scatterplot(data=df_eda, x='IAA', y='IDA', ax=ax2, alpha=0.6, color='orange')
ax2.set_title('Autoavaliação (IAA) vs Aprendizagem (IDA)', fontsize=14)

plt.show()

> **Insight:** O gráfico de Engajamento vs Aprendizagem mostra uma tendência positiva clara: alunos mais engajados tendem a ter melhor desempenho acadêmico. Já a Autoavaliação apresenta uma dispersão maior, indicando que a percepção do aluno nem sempre reflete diretamente sua nota.

## 6. Análise de Adequação de Nível (IAN)

Verificando a variabilidade e possíveis outliers no Indicador de Adequação de Nível.

In [ ]:
plt.figure(figsize=(8, 6))
sns.boxplot(y=df_eda['IAN'], color='lightgreen')
plt.title('Boxplot do Indicador de Adequação de Nível (IAN)', fontsize=14)
plt.ylabel('Valor do IAN')
plt.show()

> **Insight:** O IAN apresenta uma concentração em valores altos, mas com alguns outliers na parte inferior, representando alunos que podem estar em níveis de ensino inadequados para sua idade ou conhecimento prévio.